In [1]:
%%capture
!pip install unsloth
# Pull latest Unsloth optimizations for Colab T4
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [2]:
!pip install torch

In [3]:
import torch
from unsloth import FastLanguageModel

max_seq_length = 1024  # Tickets are ~30-60 words, so 1024 is plenty and saves VRAM
dtype = None           # Auto-detects float16 for Tesla T4
load_in_4bit = True    # Use 4-bit QLoRA quantization

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# Attach LoRA adapters to all linear layers
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.10.8: Fast Llama patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-bnb-4bit as a legacy tokenizer.
Unsloth 2026.10.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: compiling each transformer block as one region (LlamaDecoderLayer).


In [4]:
def inspect_model(model, tokenizer, max_seq_length):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    print("=" * 55)
    print("MODEL SUMMARY")
    print("=" * 55)

    print("Model type:", model.config.model_type)
    print("Hidden size:", model.config.hidden_size)
    print("Layers:", model.config.num_hidden_layers)
    print("Attention heads:", model.config.num_attention_heads)

    print("\nPARAMETERS")
    print(f"Total: {total:,}")
    print(f"Trainable: {trainable:,}")
    print(f"Frozen: {total - trainable:,}")

    if total:
        print(f"Trainable %: {100 * trainable / total:.4f}%")

    print("\nTOKENIZER")
    print("Vocabulary:", len(tokenizer))
    print("Sequence length:", max_seq_length)
    print("EOS token:", tokenizer.eos_token)

    print("\nADAPTERS")
    if hasattr(model, "print_trainable_parameters"):
        model.print_trainable_parameters()

    print("\nGPU")
    if torch.cuda.is_available():
        print("Device:", torch.cuda.get_device_name(0))
        print(
            "Allocated GB:",
            round(torch.cuda.memory_allocated() / 1024**3, 2)
        )
        print(
            "Reserved GB:",
            round(torch.cuda.memory_reserved() / 1024**3, 2)
        )
    else:
        print("CUDA is unavailable")

    print("=" * 55)


inspect_model(model, tokenizer, max_seq_length)

MODEL SUMMARY
Model type: llama
Hidden size: 3072
Layers: 28
Attention heads: 24

PARAMETERS
Total: 1,827,777,536
Trainable: 24,313,856
Frozen: 1,803,463,680
Trainable %: 1.3302%

TOKENIZER
Vocabulary: 128256
Sequence length: 1024
EOS token: <|eot_id|>

ADAPTERS
trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511

GPU
Device: Tesla T4
Allocated GB: 2.19
Reserved GB: 2.21


In [6]:
import json
import pandas as pd
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

# Apply official Llama 3.1/3.2 Instruct chat template
tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")

# Load the uploaded CSV
df_train = pd.read_csv("/content/finetune_train_1000.csv")
df_train["conversations"] = df_train["messages_json"].apply(json.loads)

hf_dataset = Dataset.from_pandas(df_train[["conversations"]])

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(
            convo, tokenize=False, add_generation_prompt=False
        )
        for convo in convos
    ]
    return {"text": texts}

dataset = hf_dataset.map(formatting_prompts_func, batched=True)
print("Sample formatted training example:\n")
print(dataset[0]["text"])

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Sample formatted training example:

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

You are an automated IT & Customer Support Ticket Triage AI. Analyze the support ticket and return ONLY a valid JSON object with the exact keys: "category" (Billing, Technical Bug, Account Access, Feature Request, Security & Compliance, General Inquiry), "urgency" (Low, Medium, High, Critical), and "department" (Finance, Tier-1 Support, Engineering, Security).<|eot_id|><|start_header_id|>user<|end_header_id|>

Customer Msg: Hi! What are your customer support operating hours for the `us-west-2` timezone, and do you offer onboarding webinars for non-profit teams?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{"category": "General Inquiry", "urgency": "Low", "department": "Tier-1 Support"}<|eot_id|>


In [7]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=15,
        num_train_epochs=2,          # 2 full passes over the 1,000 tickets (~250 steps)
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="outputs",
        report_to="none",
    ),
)

# CRITICAL: Only compute loss on the Assistant's JSON output, not the user's ticket text!
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
)

# Start training!
trainer_stats = trainer.train()

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,000 | Num Epochs = 2 | Total steps = 250
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,0.107305
20,0.027525
30,0.014812
40,0.004045
50,0.001830
60,0.006933
70,0.015332
80,0.003735
90,0.000897
100,0.000163


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-250/tokenizer_config.json.


In [8]:
FastLanguageModel.for_inference(model)

SYSTEM_PROMPT = df_train["system_prompt"].iloc[0]
test_ticket = (
    "URGENT 911 EMERGENCY!! 🚨 When I click the 'Download Billing Invoice' button "
    "in Chrome, the tab crashes with TypeError: undefined. My credit card charge is fine, "
    "fix the broken UI button!"
)

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": test_ticket},
]

# Tokenize and create attention mask on GPU
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to("cuda")

# Use do_sample=False for deterministic (greedy) decoding instead of temperature=0.0
outputs = model.generate(
    **inputs,
    max_new_tokens=64,
    do_sample=False,
    use_cache=True,
)

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)
print("Fine-Tuned Model Output:\n", response)

Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Fine-Tuned Model Output:
 {"category": "Technical Bug", "urgency": "High", "department": "Engineering"}


In [9]:
from google.colab import drive
import shutil

# 1. Merge LoRA adapters and quantize to 4-bit GGUF (takes ~4-5 mins)
model.save_pretrained_gguf(
    "ticket_classifier_gguf",
    tokenizer,
    quantization_method="q4_k_m",
)

# 2. Also save Unsloth's auto-generated Ollama Modelfile!
print("\nAuto-generated Ollama Modelfile:\n")
with open("ticket_classifier_gguf/Modelfile", "r") as f:
    print(f.read())

# # 3. Mount Google Drive and copy the GGUF + Modelfile for fast download
# drive.mount("/content/drive")
# shutil.copy("ticket_classifier_gguf/unsloth.Q4_K_M.gguf", "/content/drive/MyDrive/llama3.2-3b-ticket-ft.Q4_K_M.gguf")
# shutil.copy("ticket_classifier_gguf/Modelfile", "/content/drive/MyDrive/Modelfile")
# print("\nSuccessfully saved GGUF and Modelfile to your Google Drive!")

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.


Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in ticket_classifier_gguf/tokenizer_config.json.


...

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [03:00<00:00, 90.45s/it]


Unsloth: Merge process complete. Saved to `/content/ticket_classifier_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b11798-mix-f8a7950 (app-b11798-mix-f8a7950-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['ticket_classifier_gguf_gguf/Llama-3.2-3B-Instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...


Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### We removed it in GGUF's chat template for you.


Unsloth: All GGUF conversions completed successfully!
Generated files: ['ticket_classifier_gguf_gguf/Llama-3.2-3B-Instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model ticket_classifier_gguf_gguf/Llama-3.2-3B-Instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to ticket_classifier_gguf_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f ticket_classifier_gguf_gguf/Modelfile

Auto-generated Ollama Modelfile:



FileNotFoundError: [Errno 2] No such file or directory: 'ticket_classifier_gguf/Modelfile'

In [ ]:
from google.colab import drive
import shutil

# 1. Print the auto-generated Ollama Modelfile from the actual folder
with open("ticket_classifier_gguf_gguf/Modelfile", "r") as f:
    print(f.read())

# 2. Mount Google Drive and copy the exact generated files
drive.mount("/content/drive")

shutil.copy(
    "ticket_classifier_gguf_gguf/Llama-3.2-3B-Instruct.Q4_K_M.gguf",
    "/content/llama3.2-3b-ticket-ft.Q4_K_M.gguf"
)
shutil.copy(
    "ticket_classifier_gguf_gguf/Modelfile",
    "/content/Modelfile"
)

print("\n✅ Successfully copied llama3.2-3b-ticket-ft.Q4_K_M.gguf and Modelfile to Google Drive!")

FileNotFoundError: [Errno 2] No such file or directory: 'ticket_classifier_gguf_gguf/Modelfile'

In [ ]:
7